In [3]:
import os
import numpy as np
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow import keras


def load_data():
    label = ["paper", "rock", "scissors"]
    X_data = []
    Y_data = []
    root = r'data2'

    print("Reading Data...")
    for i, cls in enumerate(label):
        print(f"Opening {cls}/")
        class_path = os.path.join(root, cls)

        if not os.path.exists(class_path):
            print(f"[경고] 경로를 찾을 수 없습니다: {class_path}")
            continue

        for el in os.listdir(class_path):
            img_path = os.path.join(class_path, el)
            try:
                with Image.open(img_path) as img:
                    img = img.convert('RGB')
                    img = img.resize((32, 32)) # 새 모델 구조에 맞춰 32x32로 리사이즈
                    X_data.append(np.asarray(img))
                    Y_data.append(i)
            except Exception as e:
                print(f"파일 로드 실패 ({el}): {e}")

    if len(X_data) == 0:
        raise ValueError("데이터셋 폴더에 이미지가 없거나 경로가 잘못되었습니다.")

    X_data = np.asarray(X_data)
    Y_data = np.asarray(Y_data)

    train_X, test_X, train_Y, test_Y = train_test_split(
        X_data, Y_data, test_size=0.2, random_state=42, stratify=Y_data
    )

    train_X = train_X / 255.0
    test_X = test_X / 255.0

    print("\n\nData Read Done!")
    print(f"Training X Size : {train_X.shape}")
    print(f"Training Y Size : {train_Y.shape}")
    print(f"Test X Size : {test_X.shape}")
    print(f"Test Y Size : {test_Y.shape}\n\n")

    return train_X, train_Y, test_X, test_Y

train_X, train_Y, test_X, test_Y = load_data()

# Convert labels to one-hot encoding
train_Y = tf.keras.utils.to_categorical(train_Y, num_classes=3)
test_Y = tf.keras.utils.to_categorical(test_Y, num_classes=3)

# 3. 고성능 CNN 인공신경망 제작 (오타 수정 및 구조 완성)
EPOCHS = 20
model = keras.Sequential([
    # CNN의 첫 레이어에는 입력 이미지의 크기(32x32)와 채널 수(3)를 명시합니다.
    keras.layers.Input(shape=(32, 32, 3)),

    # 블록 1
    keras.layers.Conv2D(32, (3, 3), padding='same'), # 크기 유지를 위해 same 패딩 추가
    keras.layers.BatchNormalization(),
    keras.layers.ReLU(),
    keras.layers.MaxPooling2D((2, 2)),

    # 블록 2
    keras.layers.Conv2D(64, (3, 3), padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.ReLU(),
    keras.layers.MaxPooling2D((2, 2)),

    # 블록 3
    keras.layers.Conv2D(64, (3, 3), padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.ReLU(),

    # 분류기 (Dense 층 연결)
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(rate=0.5),                  # 과적합 방지
    keras.layers.Dense(3, activation="softmax")      # 가위, 바위, 보 3개 클래스에 맞게 수정
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train model
history = model.fit(
    train_X, train_Y,
    batch_size=32,
    epochs=EPOCHS,
    validation_data=(test_X, test_Y)
)

# Save model
model.save('rps_model.h5')
print("Model saved to rps_model.h5")

Reading Data...
Opening paper/
Opening rock/
Opening scissors/


Data Read Done!
Training X Size : (2319, 32, 32, 3)
Training Y Size : (2319,)
Test X Size : (580, 32, 32, 3)
Test Y Size : (580,)


Epoch 1/20
73/73 ━━━━━━━━━━━━━━━━━━━━ 20s 112ms/step - accuracy: 0.7727 - loss: 0.6796 - val_accuracy: 0.3603 - val_loss: 0.9835
Epoch 2/20
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9715 - loss: 0.0917 - val_accuracy: 0.5914 - val_loss: 0.9369
Epoch 3/20
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9918 - loss: 0.0329 - val_accuracy: 0.6690 - val_loss: 0.8939
Epoch 4/20
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9948 - loss: 0.0212 - val_accuracy: 0.8259 - val_loss: 0.5256
Epoch 5/20
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9944 - loss: 0.0186 - val_accuracy: 0.9931 - val_loss: 0.0312
Epoch 6/20
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9953 - loss: 0.0141 - val_accuracy: 0.9931 - val_loss: 0.0375
Epoch 7/20
73/73 ━━━━━━━━━━━━━━━━━━━━ 1

Model saved to rps_model.h5
